# Modul 06: Datenaufteilung, Baselines, Verluste und Metriken | Übungen

## Überblick

Sie erstellen reproduzierbare Train-, Validierungs- und Testaufteilungen und berücksichtigen dabei Klassenverteilung, Gruppen und Zeitordnung. Anschließend berechnen Sie Regressions- und Klassifikationsmetriken manuell, vergleichen Baselines und erkennen typische Datenleckage.

**Zugehörige Vorlesungen**

- **Daten aufteilen**
- **Verluste und Metriken**

## Lernziele

Nach der Bearbeitung können Sie:

- reproduzierbare Splits mit Indizes erstellen und Sonderstrukturen wie Klassen, Gruppen und Zeit berücksichtigen.
- Datenleckage durch Zielinformationen, überlappende Gruppen oder falsch angepasste Vorverarbeitung erkennen.
- Regressionsverluste, Baselines, Konfusionsmatrix, Precision, Recall, F1 und probabilistische Verluste berechnen.

## Geprüfte Fähigkeiten

- Indexbasierte, stratifizierte, gruppenbasierte und zeitliche Splits
- leakage-freie Skalierung und dokumentierte Splitkontrollen
- manuelle Metrikberechnung und fachlich passende Baselines

## Hinweise zur Bearbeitung

Dieses Notebook dient als praktische Übung und Lernstandskontrolle. Führen Sie zuerst die Einrichtungszelle aus und bearbeiten Sie danach die Aufgaben in der angegebenen Reihenfolge. Die vorgesehenen Arbeitsbereiche sind deutlich markiert.

- **Erwarteter Schwierigkeitsgrad:** mittel
- Verwenden Sie sprechende Variablennamen und prüfen Sie wichtige Zwischenformen und Wertebereiche.
- Verändern Sie die vorgegebenen Zufalls-Startwerte nur, wenn eine Aufgabe dies ausdrücklich verlangt.
- Interpretieren Sie Ergebnisse fachlich. Eine einzelne Kennzahl ist selten eine vollständige Begründung.
- Alle Aufgaben sind für die kostenlose Google-Colab-Umgebung ausgelegt. Die Datensätze und Modelle sind bewusst klein gehalten. Eine GPU ist nicht erforderlich, kann aber bei einzelnen Deep-Learning-Aufgaben die Laufzeit verkürzen.

## Einrichtung und gemeinsame Datenbasis

Die Setup-Zelle erzeugt kleine tabellarische Daten mit wiederholten Patientengruppen, Zeitordnung und unausgewogener Zielvariable. Zusätzlich stehen feste Regressions- und Klassifikationsvorhersagen bereit.

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score
from sklearn.model_selection import GroupShuffleSplit, train_test_split
from sklearn.preprocessing import StandardScaler

RANDOM_SEED = 42
rng = np.random.default_rng(RANDOM_SEED)

# 45 Gruppen mit je drei wiederholten Messungen.
anzahl_gruppen = 45
messungen_pro_gruppe = 3
gruppen_id = np.repeat(np.arange(1001, 1001 + anzahl_gruppen), messungen_pro_gruppe)
zeitpunkt = np.tile(np.arange(messungen_pro_gruppe), anzahl_gruppen)

alter_basis = rng.integers(25, 75, size=anzahl_gruppen)
alter = np.repeat(alter_basis, messungen_pro_gruppe) + zeitpunkt
marker = rng.normal(0, 1, size=len(gruppen_id)) + 0.025 * (alter - 50)
puls = rng.normal(72, 8, size=len(gruppen_id)) + 4 * marker

# Seltenes positives Ereignis, das von Alter und Marker abhängt.
logit = -2.5 + 0.045 * (alter - 50) + 0.9 * marker
wahrscheinlichkeit = 1 / (1 + np.exp(-logit))
ziel = rng.binomial(1, wahrscheinlichkeit)

patienten = pd.DataFrame(
    {
        "gruppen_id": gruppen_id,
        "zeitpunkt": zeitpunkt,
        "alter": alter.astype(float),
        "marker": marker,
        "puls": puls,
        "ziel": ziel,
    }
)

# Feste Regressionsdaten für manuelle Verlustberechnungen.
y_reg_true = np.array([10.0, 12.0, 14.0, 18.0, 25.0, 40.0])
y_reg_model = np.array([11.0, 11.5, 16.0, 17.0, 23.0, 31.0])

# Feste Wahrscheinlichkeiten für eine binäre Klassifikation.
y_cls_true = np.array([0, 0, 1, 1, 0, 1, 0, 1, 1, 0])
y_cls_score = np.array([0.10, 0.35, 0.62, 0.91, 0.55, 0.74, 0.20, 0.48, 0.83, 0.05])

print("Einrichtung abgeschlossen.")
print("Patiententabelle:", patienten.shape)
print("Positive Klasse:", f"{patienten['ziel'].mean():.1%}")

### Aufgabe 1: Train-, Validierungs- und Testindizes manuell erzeugen

Verwenden Sie die Zeilenindizes von `patienten`:

1. Mischen Sie die Indizes reproduzierbar mit einem lokalen Zufallsgenerator.
2. Teilen Sie 60 % Training, 20 % Validierung und 20 % Test zu.
3. Prüfen Sie Splitgrößen, vollständige Abdeckung und fehlende Überschneidungen.
4. Speichern Sie die Teilmengen als `train_manuell`, `valid_manuell` und `test_manuell`.
5. Dokumentieren Sie den verwendeten Seed und die tatsächlichen Größen in einem DataFrame.

In [ ]:
alle_indizes = patienten.index.to_numpy()

# ============================================================
# IHRE LÖSUNG HIER
# ============================================================

> **Ihre Antwort:**
>
> Welche wichtige Datenstruktur berücksichtigt dieser zufällige Split noch nicht?

### Aufgabe 2: Stratifizierte, gruppenbasierte und zeitliche Splits vergleichen

Erstellen Sie drei alternative Splits:

1. **Stratifiziert:** 75 % Training und 25 % Test mit ähnlichem Klassenanteil.
2. **Gruppenbasiert:** ganze `gruppen_id` im Verhältnis ungefähr 75/25 trennen.
3. **Zeitlich:** alle Zeilen mit `zeitpunkt < 2` als Training und `zeitpunkt == 2` als Test.

Berichten Sie für jeden Split Zeilenzahl, positiven Anteil und gegebenenfalls Anzahl überlappender Gruppen. Erklären Sie, für welche Einsatzannahme jeder Split geeignet ist.

In [ ]:
merkmale = ["alter", "marker", "puls"]
X_patienten = patienten[merkmale]
y_patienten = patienten["ziel"]

# ============================================================
# IHRE LÖSUNG HIER
# ============================================================

> **Ihre Antwort:**
>
> Ordnen Sie jedem Split eine realistische Einsatzfrage zu.

### Aufgabe 3: Datenleckage durch Zielinformation und Skalierung erkennen

1. Erzeugen Sie absichtlich ein unzulässiges Merkmal `ziel_kopie = ziel` und erklären Sie, warum es Zielwertleckage ist.
2. Verwenden Sie den gruppenbasierten Split aus Aufgabe 2.
3. Vergleichen Sie zwei Skalierungen:
   - falsch: `StandardScaler` auf allen Daten anpassen,
   - korrekt: nur auf Trainingsdaten anpassen und danach Testdaten transformieren.
4. Geben Sie die gelernten Mittelwerte beider Skalierer aus.
5. Trainieren Sie ein logistisches Modell ausschließlich mit den korrekten Merkmalen und der korrekten Skalierung.

In [ ]:
patienten_leakage = patienten.copy()
patienten_leakage["ziel_kopie"] = patienten_leakage["ziel"]

# ============================================================
# IHRE LÖSUNG HIER
# ============================================================

> **Ihre Antwort:**
>
> Warum kann selbst eine Vorverarbeitung ohne Zielspalte Datenleckage verursachen?

### Aufgabe 4: Regressionsverluste und konstante Baselines manuell berechnen

Berechnen Sie ausschließlich mit NumPy:

1. Residuen `Vorhersage - Istwert`.
2. MAE, MSE und RMSE für `y_reg_model`.
3. Eine Mittelwert-Baseline, deren Konstante nur aus den ersten vier Zielwerten gelernt wird.
4. Eine Median-Baseline mit derselben Trainingsmenge.
5. MAE und RMSE beider Baselines auf den letzten zwei Werten.
6. Eine Vergleichstabelle und eine kurze Interpretation der großen letzten Abweichung.

In [ ]:
y_reg_train = y_reg_true[:4]
y_reg_test = y_reg_true[4:]
modell_test_vorhersage = y_reg_model[4:]

# ============================================================
# IHRE LÖSUNG HIER
# ============================================================

> **Ihre Antwort:**
>
> Warum reagiert RMSE stärker auf den großen Fehler beim Zielwert 40 als MAE?

### Aufgabe 5: Konfusionsmatrix, Schwellenwerte und Wahrscheinlichkeitsverlust

Schreiben Sie eine Funktion `klassifikationsbericht(y_true, scores, schwelle)`, die manuell berechnet:

- vorhergesagte Labels,
- TP, TN, FP und FN,
- Accuracy, Precision, Recall und F1,
- binäre Kreuzentropie mit numerischer Absicherung.

Vergleichen Sie die Schwellenwerte 0,50 und 0,70 in einer Tabelle.

In [ ]:
def klassifikationsbericht(y_true, scores, schwelle):
    """Berechnet binäre Kennzahlen ohne sklearn-Metrikfunktionen."""
    pass

# ============================================================
# IHRE LÖSUNG HIER
# ============================================================

> **Ihre Antwort:**
>
> Warum ändert sich die Kreuzentropie in dieser Tabelle nicht mit dem Schwellenwert?

### Aufgabe 6: Integrationsaufgabe: gruppensichere Bewertung mit Baseline

Nutzen Sie den gruppenbasierten Split und erstellen Sie einen vollständigen kleinen Bewertungsablauf:

1. Mehrheitsklassen-Baseline aus `y_train`.
2. Leakage-freie Skalierung.
3. Logistische Regression.
4. Wahrscheinlichkeiten und Vorhersagen bei Schwelle 0,50.
5. Manuelle Kennzahlen mit Ihrer Funktion.
6. Vergleich mit der Baseline.
7. Splitprotokoll mit Gruppenanzahl, Klassenanteilen und Seed.

Begründen Sie, ob Accuracy bei der vorliegenden Klassenverteilung ausreicht.

In [ ]:
# X_train, X_test, y_train und y_test stammen aus Aufgabe 3.

# ============================================================
# IHRE LÖSUNG HIER
# ============================================================

> **Ihre Antwort:**
>
> Bewerten Sie Modell und Baseline mit besonderem Blick auf die seltene positive Klasse.

## Abschlusskontrolle

Prüfen Sie vor dem Abschluss:

- Lassen sich alle Zellen in sinnvoller Reihenfolge ausführen?
- Sind Formen, Datentypen, Wertebereiche und Zufalls-Startwerte dokumentiert?
- Wurden Trainings-, Validierungs- und Testinformationen sauber getrennt?
- Sind Diagramme und Kennzahlen beschriftet und fachlich interpretiert?
- Können Sie erklären, warum die gewählten Methoden zur Aufgabenstellung passen?